# Vector Graph Memory API - Playground

This notebook demonstrates using the Vector Graph Memory API for a job search use case.

**Prerequisites:**
```bash
docker compose up -d
```

## Setup

In [1]:
import requests
import json

# API Configuration
API_BASE_URL = "http://localhost:8000"
SESSION_ID = "playground-session-1"

def chat(message: str, session_id: str = SESSION_ID) -> dict:
    """Send a chat message to the API."""
    response = requests.post(
        f"{API_BASE_URL}/v1/chat/completions",
        json={
            "model": "vector-graph-memory",
            "messages": [{"role": "user", "content": message}],
            "user": session_id,
        }
    )
    response.raise_for_status()
    return response.json()

def get_proposals(session_id: str = SESSION_ID) -> dict:
    """Get pending memory proposals for the session."""
    response = requests.get(f"{API_BASE_URL}/memory/proposals/{session_id}")
    response.raise_for_status()
    return response.json()

def confirm_proposal(proposal_id: str, session_id: str = SESSION_ID, action: str = "add_new") -> dict:
    """Confirm a memory proposal."""
    response = requests.post(
        f"{API_BASE_URL}/memory/confirm/{session_id}/{proposal_id}",
        params={"action": action}
    )
    response.raise_for_status()
    return response.json()

def get_audit_log(session_id: str = SESSION_ID) -> dict:
    """Get audit log for the session."""
    response = requests.get(f"{API_BASE_URL}/memory/audit/{session_id}")
    response.raise_for_status()
    return response.json()

# Test API connection
response = requests.get(f"{API_BASE_URL}/")
if response.status_code == 200:
    print("✓ Connected to Vector Graph Memory API")
    print(f"  Service: {response.json()['service']}")
    print(f"  Version: {response.json()['version']}")
else:
    print("✗ Failed to connect to API")

✓ Connected to Vector Graph Memory API
  Service: vector-graph-memory-api
  Version: 1.0.0


## Example 1: Basic Conversation

In [2]:
result = chat("What tools do you have available for managing my job search?")
print(result["choices"][0]["message"]["content"])

I have a few tools for managing your job search that involve memory management. Here are the key capabilities:

1. **Search Memory**: I can search for existing relevant information from our previous conversations that might assist your job search.
2. **Propose Memory Addition**: I can suggest adding new information to our memory, such as job preferences or companies you're interested in, which can help keep track of your search progress.
3. **Get Memory Context**: I can retrieve context about previous conversations or memory items we've discussed to provide you a comprehensive view related to your job search.

If there's anything specific you'd like me to assist with or retain in memory, just let me know!


## Example 2: Adding Information to Memory

The agent will propose adding items to memory when you use certain trigger phrases or patterns.

In [3]:
# Tell the agent about a job application
result = chat(
    "I just applied to a Senior Software Engineer position at Google. "
    "The role is focused on distributed systems and will be based in Mountain View."
)
print(result["choices"][0]["message"]["content"])

Should I add "Applied to a Senior Software Engineer position at Google focused on distributed systems, based in Mountain View" to memory?


In [4]:
# Check for pending proposals
proposals = get_proposals()
print(f"\nPending proposals: {len(proposals['proposals'])}")
for proposal_id, proposal_data in proposals["proposals"].items():
    print(f"\nProposal ID: {proposal_id}")
    print(f"  Type: {proposal_data['entity_type']}")
    print(f"  Content: {proposal_data['content'][:100]}...")


Pending proposals: 1

Proposal ID: 63710ea0-b5df-4737-93d6-2f0697855c10
  Type: job
  Content: Applied to a Senior Software Engineer position at Google focused on distributed systems, based in Mo...


In [5]:
# Confirm the first proposal
proposals = get_proposals()
if proposals["proposals"]:
    proposal_id = list(proposals["proposals"].keys())[0]
    result = confirm_proposal(proposal_id)
    print(result["message"])
else:
    print("No pending proposals")

Successfully added job to memory with ID: 2fc76ddc-b641-48c7-977b-07f733df290b


## Example 3: Querying Memory

The agent automatically searches memory when relevant to your queries.

**Note:** For multi-turn conversations, you need to maintain conversation history across requests. Each API call includes the full conversation context so the agent understands references like "the position" or "that job".

In [6]:
# Ask about stored information - maintaining conversation history
conversation = []

queries = [
    "What software engineering positions have I applied to?",
    "Tell me about the Google job",
    "Where is the position located?"
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Q: {query}")
    print(f"{'='*60}")

    # Add user message to conversation
    conversation.append({"role": "user", "content": query})

    # Send entire conversation history
    response = requests.post(
        f"{API_BASE_URL}/v1/chat/completions",
        json={
            "model": "vector-graph-memory",
            "messages": conversation,
            "user": SESSION_ID,
        }
    )
    result = response.json()

    # Add assistant response to conversation
    assistant_message = result["choices"][0]["message"]["content"]
    conversation.append({"role": "assistant", "content": assistant_message})

    print(assistant_message)


Q: What software engineering positions have I applied to?
It seems that you have applied to a Senior Software Engineer position at Google, which focuses on distributed systems and is based in Mountain View. 

Would you like to add any other specific details or updates related to your job applications, or should I add this position to your memory?

Q: Tell me about the Google job
I found a similar entry in memory regarding your application for the Senior Software Engineer position at Google, which focuses on distributed systems and is based in Mountain View. 

Would you like to update the existing entry, or should I add this information as a new memory?

Q: Where is the position located?
I found a similar entry in memory detailing your application for the Senior Software Engineer position at Google, which focuses on distributed systems and is based in Mountain View.

Would you like to update this existing entry with any additional details or modifications, or should I keep it as is?


## Example 4: Multi-Turn Conversation with Auto Memory

The agent can automatically detect when to store information based on AI determination mode.

In [7]:
# Conversation about a new opportunity
messages = [
    "I had a phone screening with Meta today",
    "The recruiter mentioned the team works on AI infrastructure",
    "They're looking for someone with distributed systems experience",
    "The position is remote-friendly with offices in Menlo Park",
    "I think this could be a good fit for my background"
]

for i, msg in enumerate(messages, 1):
    print(f"\n--- Message {i} ---")
    print(f"User: {msg}")
    result = chat(msg)
    print(f"Agent: {result['choices'][0]['message']['content']}")

# Check if any proposals were created
proposals = get_proposals()
print(f"\n{'='*60}")
if proposals["proposals"]:
    print(f"✓ Agent proposed {len(proposals['proposals'])} memory addition(s)")
    for proposal_id in proposals["proposals"].keys():
        print(f"  - {proposal_id}")
else:
    print("✗ No memory proposals generated")


--- Message 1 ---
User: I had a phone screening with Meta today
Agent: Should I add "User had a phone screening with Meta" as an event to memory?

--- Message 2 ---
User: The recruiter mentioned the team works on AI infrastructure
Agent: I have identified two additions to memory:

1. Should I add **"The team works on AI infrastructure"** to memory as a team?
2. There is a similar item in memory regarding the **Senior Software Engineer position at Google**. Would you like me to add this as a new job, or update the existing one with the information about AI infrastructure?

Please let me know how you’d like to proceed!

--- Message 3 ---
User: They're looking for someone with distributed systems experience
Agent: I found a similar memory item related to applying for a Senior Software Engineer position at Google, which focused on distributed systems. Would you like to update this existing entry with the new information about them looking for someone with distributed systems experience, o

## Example 5: Viewing Audit History

In [8]:
# Get audit log for this session
audit = get_audit_log()
print(f"Audit log entries for session '{SESSION_ID}': {len(audit['entries'])}\n")

for entry in audit["entries"]:
    print(f"[{entry['timestamp']}] {entry['operation']}")
    print(f"  {entry['summary']}")
    print(f"  Entities: {entry['entities']}")
    print()

Audit log entries for session 'playground-session-1': 3

[2026-03-22T02:33:30.768297] add_node
  Added tools: The tools available for managing a job search include resources for job listings, resume building, i
  Entities: ['51164d38-94fa-416c-9ee0-7ca0287d5b43']

[2026-03-22T02:41:56.073909] add_node
  Added job: Applied to a Senior Software Engineer position at Google focused on distributed systems based in Mou
  Entities: ['e3451c47-61b8-4536-83ff-c71bbda5e3d9']

[2026-03-22T02:45:44.449450] add_node
  Added job: Applied to a Senior Software Engineer position at Google focused on distributed systems, based in Mo
  Entities: ['2fc76ddc-b641-48c7-977b-07f733df290b']



## Example 6: Testing Different Query Types

See how the agent handles various types of queries and determines what needs to be remembered.

In [9]:
test_scenarios = [
    {
        "name": "Factual Information",
        "message": "The interview is scheduled for next Tuesday at 2pm"
    },
    {
        "name": "Contact Information",
        "message": "The recruiter's name is Sarah Johnson, email: sjohnson@meta.com"
    },
    {
        "name": "Status Update",
        "message": "I passed the technical interview!"
    },
    {
        "name": "Casual Query",
        "message": "What's the weather like?"
    },
]

for scenario in test_scenarios:
    print(f"\n{'='*60}")
    print(f"Scenario: {scenario['name']}")
    print(f"{'='*60}")
    print(f"User: {scenario['message']}")
    
    result = chat(scenario["message"])
    print(f"Agent: {result['choices'][0]['message']['content']}")
    
    # Check if a proposal was created
    proposals_before = get_proposals()
    print(f"Proposals: {len(proposals_before['proposals'])}")


Scenario: Factual Information
User: The interview is scheduled for next Tuesday at 2pm
Agent: Yes, please add "Interview scheduled for next Tuesday at 2pm" to memory.
Proposals: 6

Scenario: Contact Information
User: The recruiter's name is Sarah Johnson, email: sjohnson@meta.com
Agent: I found two items that can be added to memory regarding Sarah Johnson: 

1. Recruiter: Sarah Johnson, Email: sjohnson@meta.com
2. Email of Sarah Johnson: sjohnson@meta.com (as a contact)

Should I add both of these to memory?
Proposals: 8

Scenario: Status Update
User: I passed the technical interview!
Agent: Yes, should I add "Passed the technical interview" as an event to memory?
Proposals: 9

Scenario: Casual Query
User: What's the weather like?
Agent: It seems there are no relevant entries in memory related to the weather. If you have specific details or experiences about the weather you'd like me to remember, please let me know! Would you like to add something specific about the weather to memory?

## Helper Functions

Additional utilities for working with the API.

In [10]:
def print_response(result: dict):
    """Pretty print a chat response."""
    print(f"Model: {result['model']}")
    print(f"Message: {result['choices'][0]['message']['content']}")
    print(f"Tokens: {result['usage']['total_tokens']} total")
    print(f"  (prompt: {result['usage']['prompt_tokens']}, completion: {result['usage']['completion_tokens']})")

def confirm_all_proposals(session_id: str = SESSION_ID):
    """Confirm all pending proposals for a session."""
    proposals = get_proposals(session_id)
    confirmed = []
    
    for proposal_id in proposals["proposals"].keys():
        result = confirm_proposal(proposal_id, session_id)
        confirmed.append((proposal_id, result["message"]))
    
    return confirmed

# Example usage
print("✓ Helper functions loaded")

✓ Helper functions loaded
